<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_8/Day_3/Exercises%20/Exercises_XP_RAG_w8_d3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])


## 2) Split into chunks


In [ ]:
chunk_size = 512
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
    strip_whitespace=True
)

splits = splitter.split_documents(documents)

print("Chunks:", len(splits))
print("First chunk metadata:", splits[0].metadata)
print("-" * 30)
print("First chunk content:\n", splits[0].page_content[:350])


## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

## 4) Build the RAG chain


In [ ]:
!pip install langchain

In [ ]:
!pip install -U langchain-classic

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

llm_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(llm_id)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_id)

hf = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

print("RAG chain ready")

In [ ]:
question = "How can I use the Hugging Face hub?"
response = qa.invoke(question)
print(response["result"])

## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

rag_result = qa.invoke({"query": q})

print("Q:", q)
print("\n--- No-RAG answer ---\n", no_rag_answer)
print("\n--- RAG answer ---\n", rag_result["result"])
print("\nSources:")

if "source_documents" in rag_result:
    for d in rag_result["source_documents"]:
        print("-", d.metadata.get("source"))
else:
    print("No sources returned.")

**6. Retrieval sanity check (required)**

In [ ]:
test_query = "What is the command to log into the Hugging Face Hub?"

retrieved_chunks = retriever.invoke(test_query)

print(f"--- Sanity Check for Query: '{test_query}' ---\n")
print(f"Number of chunks retrieved: {len(retrieved_chunks)}")

for i, doc in enumerate(retrieved_chunks):
    print(f"\n[Chunk {i+1}]")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content[:300]}...")
    print("-" * 20)

**7. RAG question answering**

In [ ]:
test_questions = [
    "How do I log into the Hugging Face Hub in a notebook?",
    "What library makes the Hugging Face integration possible?",
    "How can I share models in the Hub?"
]

print("--- Starting RAG Evaluation ---\n")

for i, question in enumerate(test_questions):
    print(f"Test {i+1}: {question}")
    result = qa.invoke({"query": question})

    print(f"Response: {result['result']}")

    print("Sources detected:")
    sources = set([doc.metadata.get("source") for doc in result.get("source_documents", [])])
    for s in sources:
        print(f"  - {s}")

    print("-" * 30 + "\n")

print("Exercise XP Complete!")